# Assistiv Systems — FEP Real Data Fetcher
**Frailty Geography Intelligence · Layer 2 · Kent & Medway · v4.0**

Fetches real NHS data and commits `kent-fep-data.json` to GitHub:
- **NHS Fingertips** — 10 outcomes indicators via `fingertips_py`
- **NHSBSA English Prescribing Dataset** — 7 prescribing signal groups at practice level
- **UKHSA/Met Office Weather-Health Alert** — live heat/cold alert status for South East England ← NEW v4

### What's new in v4.0 — Weather-Health Alert Integration
The UKHSA/Met Office Weather-Health Alerting system publishes live heat and cold alerts
for England, specifically targeting the elderly and those on multiple medications.
v4.0 fetches the current alert status for the South East region and:
1. Stores it in `kent-fep-data.json` under `weather_alert`
2. Applies a **time-limited FEP uplift** to high-risk districts during amber/red alerts
3. Flags **medication risk elevation** for anticholinergic and diuretic signals
4. Provides a **Layer 4 context string** injected into RESILIENCE screening sessions

### Before running:
1. Add your GitHub token to Colab Secrets (🔑 left sidebar)
   - Name: `GITHUB_TOKEN` · Value: your token with `public_repo` scope
2. `Runtime → Run all` — takes approximately 5–8 minutes

### To refresh quarterly (EPD) / on-demand (weather):
- Update `EPD_URL` and `EPD_PERIOD` each quarter
- Weather cell can be re-run independently at any time
- Latest EPD: https://opendata.nhsbsa.net/dataset/english-prescribing-dataset-epd-with-snomed-code

---
*NHSBSA Copyright 2026 · OpenPrescribing.net, Bennett Institute, University of Oxford · OHID · UKHSA · Met Office*

## Cell 1 — Configuration
Installs dependencies and sets all constants.
Update `EPD_URL` and `EPD_PERIOD` each quarter.

**v4.0 additions:**
- `WEATHER_API_URL` — UKHSA Weather-Health Alert JSON feed (free, no key required)
- `WEATHER_MED_RISK_SIGNALS` — which EPD signals attract elevated risk during alerts
- `WEATHER_UPLIFT` — FEP score uplift applied to high-risk districts during amber/red alerts

In [ ]:
import subprocess
subprocess.run(["pip", "install", "fingertips_py", "requests", "pandas", "--quiet"], check=True)
print("Dependencies installed")

import requests, pandas as pd, fingertips_py as ftp
import json, csv, base64
from datetime import datetime, timezone
from collections import defaultdict
from google.colab import userdata

GITHUB_REPO  = "silegrand/assistivagents"
GITHUB_FILE  = "kent-fep-data.json"
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN').split('\n')[0].strip()
print(f"GitHub token: {'Found' if GITHUB_TOKEN else 'MISSING - add to Colab Secrets'}")

KENT_ICB_ONS   = "E54000032"
KENT_ICB_CODE  = "QKS"
KENT_COUNTY    = "E10000016"
ENGLAND        = "E92000001"

# ── UPDATE THESE EACH QUARTER ─────────────────────────────────────────────────
EPD_URL    = "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/6b52baa3-c550-4b1d-8cd2-46d40d840353/download/epd_snomed_202603.csv"
EPD_PERIOD = "Mar 2026"

# ── v4.0: UKHSA Weather-Health Alert ──────────────────────────────────────────
# The UKHSA/Met Office Weather-Health Alerting system publishes regional alert
# status as a public JSON feed. No API key required.
# Heat-Health Alerts run 1 Jun – 30 Sep; Cold-Health Alerts run 1 Nov – 30 Mar.
# South East England region code = 'E' in UKHSA taxonomy.
# Alert levels: green | yellow | amber | red
# Reference: https://www.metoffice.gov.uk/weather/warnings-and-advice/seasonal-advice/heat-health-alert-service
WEATHER_API_URL = "https://api.ukhsa.gov.uk/weather-health-alerting/alerts"
WEATHER_REGION  = "South East England"  # Matches Kent & Medway geography

# Signals with elevated clinical risk during heat (H) or cold (C) alerts.
# These flags are attached to the weather_alert block in the JSON output
# and injected into the RESILIENCE Layer 4 screening context string.
# Evidence base:
#   Heat → anticholinergic burden: impairs thermoregulation (sweating), raises toxicity risk
#   Heat → diuretics/ACE-ARB: accelerates dehydration, electrolyte disturbance
#   Cold → hypnotics/anxiolytics: CNS sedation increases fall risk on icy surfaces
#   Cold → diuretics: cold diuresis + electrolyte disruption in frail older adults
WEATHER_MED_RISK_SIGNALS = {
    "heat": ["anxiolytics", "bladder_antimusc", "antidepressants", "diuretics", "ace_arb"],
    "cold": ["hypnotics", "anxiolytics", "diuretics"],
}

# FEP score uplift applied to HIGH and CRITICAL risk districts during amber/red alerts.
# Rationale: an amber cold alert in Thanet (FEP 63) is a materially different event
# than in Tunbridge Wells (FEP 34). The uplift accelerates outreach priority.
# Uplift is time-limited — it appears in the JSON but does NOT permanently alter weights.
WEATHER_UPLIFT = {"green": 0, "yellow": 2, "amber": 5, "red": 8}  # FEP points

# England reference rates per 1,000 patients/month (OpenPrescribing 2024/25)
ENGLAND_PRESCRIBING_RATES = {
    'hypnotics':        10.2,
    'anxiolytics':       8.5,
    'antidepressants':  110.0,
    'bisphosphonates':   6.8,
    'diuretics':         2.5,
    'ace_arb':          95.0,
    'bladder_antimusc':  4.2,
}

# ── EPD column indices (NHSBSA EPD_SNOMED, 0-indexed) ─────────────────────────
ICB_I      = 3
BNF_I      = 14
CHEM_I     = 15
PRACTICE_I = 8
POSTCODE_I = 13
ITEMS_I    = 20
QTY_I      = 21
KENT_FILTER = 'KENT AND MEDWAY'

# ── District list sizes (NHS Digital Practice List Sizes 2024) ─────────────────
DISTRICT_LIST_SIZES = {
    "Thanet":              145_000,
    "Folkestone & Hythe":  116_000,
    "Dover":               115_000,
    "Swale":               153_000,
    "Medway":              295_000,
    "Gravesham":           106_000,
    "Ashford":             128_000,
    "Canterbury":          172_000,
    "Dartford":            111_000,
    "Maidstone":           183_000,
    "Tonbridge & Malling": 132_000,
    "Sevenoaks":           118_000,
    "Tunbridge Wells":     119_000,
}
KENT_LIST_SIZE = sum(DISTRICT_LIST_SIZES.values())
ALL_DISTRICTS  = list(DISTRICT_LIST_SIZES.keys())

# ── Postcode outward code → District lookup ────────────────────────────────────
POSTCODE_TO_DISTRICT = {
    'CT9':'Thanet','CT10':'Thanet','CT11':'Thanet','CT12':'Thanet',
    'CT1':'Canterbury','CT2':'Canterbury','CT3':'Canterbury',
    'CT4':'Canterbury','CT5':'Canterbury','CT6':'Canterbury',
    'CT13':'Dover','CT14':'Dover','CT15':'Dover','CT16':'Dover','CT17':'Dover',
    'CT18':'Folkestone & Hythe','CT19':'Folkestone & Hythe',
    'CT20':'Folkestone & Hythe','CT21':'Folkestone & Hythe','TN29':'Folkestone & Hythe',
    'ME1':'Medway','ME2':'Medway','ME3':'Medway','ME4':'Medway',
    'ME5':'Medway','ME7':'Medway','ME8':'Medway',
    'ME9':'Swale','ME10':'Swale','ME11':'Swale','ME12':'Swale','ME13':'Swale',
    'ME14':'Maidstone','ME15':'Maidstone','ME16':'Maidstone',
    'ME17':'Maidstone','ME18':'Maidstone',
    'ME19':'Tonbridge & Malling','ME20':'Tonbridge & Malling',
    'TN10':'Tonbridge & Malling','TN11':'Tonbridge & Malling','TN12':'Tonbridge & Malling',
    'DA11':'Gravesham','DA12':'Gravesham','DA13':'Gravesham',
    'DA1':'Dartford','DA2':'Dartford',
    'TN13':'Sevenoaks','TN14':'Sevenoaks','TN15':'Sevenoaks',
    'TN16':'Sevenoaks','TN8':'Sevenoaks','DA3':'Sevenoaks','DA4':'Sevenoaks',
    'TN1':'Tunbridge Wells','TN2':'Tunbridge Wells','TN3':'Tunbridge Wells',
    'TN4':'Tunbridge Wells','TN5':'Tunbridge Wells','TN6':'Tunbridge Wells',
    'TN23':'Ashford','TN24':'Ashford','TN25':'Ashford',
    'TN26':'Ashford','TN27':'Ashford','TN30':'Ashford',
}

print(f"Config v4.0 loaded | ICB: {KENT_ICB_ONS} | EPD: {EPD_PERIOD}")
print(f"District list sizes: {len(DISTRICT_LIST_SIZES)} districts, {KENT_LIST_SIZE:,} total patients")
print(f"Postcode lookup: {len(POSTCODE_TO_DISTRICT)} outward codes → {len(set(POSTCODE_TO_DISTRICT.values()))} districts")
print(f"Weather API: {WEATHER_API_URL}")

## Cell 2 — Fetch UKHSA Weather-Health Alert  ← NEW v4.0
Fetches current alert status from the UKHSA/Met Office Weather-Health Alerting system.
No API key required — public endpoint.

**What this produces:**
- `weather_alert` block in the JSON output, read by the FEP map and RESILIENCE Layer 4
- `alert_level`: green / yellow / amber / red
- `alert_type`: heat | cold | none (seasonal: heat Jun–Sep, cold Nov–Mar)
- `med_risk_signals`: list of prescribing signals with elevated clinical risk right now
- `layer4_context`: plain-English string injected into RESILIENCE screening sessions
- `fep_uplift`: integer added to FEP scores in high-risk districts during amber/red alerts

**Fallback:** if the API is unreachable, defaults to green / no alert. The rest of the
notebook runs normally — weather is additive, not load-bearing.

In [ ]:
# ── v4.0: Fetch UKHSA Weather-Health Alert ────────────────────────────────────
# The UKHSA WHA JSON API is a public endpoint — no key required.
# Falls back gracefully if unreachable.

def fetch_weather_alert():
    """Fetch current UKHSA Weather-Health Alert for South East England.
    Returns a dict with alert_level, alert_type, region, fetched_at, and derived fields.
    Fails safe to green/none if API is unreachable."""

    now_utc   = datetime.now(timezone.utc)
    month     = now_utc.month
    season    = "heat" if 6 <= month <= 9 else "cold" if month in (11,12,1,2,3) else "none"

    defaults = {
        "alert_level":      "green",
        "alert_type":       season,
        "region":           WEATHER_REGION,
        "fetched_at":       now_utc.isoformat(),
        "source":           "UKHSA Weather-Health Alerting / Met Office",
        "api_status":       "fallback",
        "med_risk_signals": [],
        "fep_uplift":       0,
        "layer4_context":   "",
    }

    try:
        resp = requests.get(
            WEATHER_API_URL,
            params={"region": WEATHER_REGION},
            timeout=10,
            headers={"Accept": "application/json", "User-Agent": "AssistivSystems/4.0"}
        )

        if resp.status_code == 200:
            data   = resp.json()
            alerts = data if isinstance(data, list) else data.get("alerts", [])

            # Find the most severe active alert for South East England
            level_order = {"red": 4, "amber": 3, "yellow": 2, "green": 1}
            best_level  = "green"
            best_type   = season

            for alert in alerts:
                region = (alert.get("region") or alert.get("area") or "").lower()
                if "south east" not in region and "kent" not in region:
                    continue
                lvl  = (alert.get("level") or alert.get("alert_level") or "green").lower()
                atype = (alert.get("type") or alert.get("alert_type") or season).lower()
                if level_order.get(lvl, 0) > level_order.get(best_level, 0):
                    best_level = lvl
                    best_type  = atype

            defaults["api_status"] = "live"
            defaults["alert_level"] = best_level
            defaults["alert_type"]  = best_type

        else:
            # API reachable but unexpected status — try scraping the UKHSA alert page
            print(f"  WHA API returned {resp.status_code} — attempting HTML fallback")
            page = requests.get(
                "https://www.ukhsa.gov.uk/weather-health-alerting",
                timeout=10, headers={"User-Agent": "AssistivSystems/4.0"}
            )
            text = page.text.lower()
            for lvl in ("red", "amber", "yellow"):
                if f"south east" in text and lvl in text:
                    defaults["alert_level"] = lvl
                    defaults["api_status"]  = "html_fallback"
                    break

    except Exception as e:
        print(f"  WHA API unreachable: {e} — defaulting to green/no alert")
        defaults["api_status"] = f"error: {str(e)[:80]}"

    # ── Derive fields from confirmed alert_level + alert_type ──────────────────
    level = defaults["alert_level"]
    atype = defaults["alert_type"]
    uplift = WEATHER_UPLIFT.get(level, 0)

    # Which medication signals have elevated clinical risk right now?
    med_risk = WEATHER_MED_RISK_SIGNALS.get(atype, []) if level in ("yellow","amber","red") else []

    # Plain-English context string for RESILIENCE Layer 4 injection
    if level == "green" or atype == "none":
        ctx = ""
    else:
        level_desc = {"yellow":"Yellow", "amber":"Amber", "red":"Red"}.get(level, level.title())
        type_desc  = "Heat-Health" if atype == "heat" else "Cold-Health"
        sigs_str   = ", ".join(med_risk) if med_risk else "none flagged"
        if atype == "heat":
            med_note = ("Anticholinergic burden (antidepressants, anxiolytics, bladder antimuscarinics) "
                        "is elevated during heat — impaired thermoregulation and dehydration risk. "
                        "Diuretic and ACE/ARB users face accelerated electrolyte depletion. "
                        "Apply heightened sensitivity to medication-related responses.")
        else:
            med_note = ("Hypnotics and anxiolytics increase fall risk on icy or wet surfaces. "
                        "Cold diuresis compounds electrolyte risk in diuretic users. "
                        "Apply heightened sensitivity to mobility, falls, and medication responses.")
        ctx = (
            f"WEATHER ALERT ACTIVE: UKHSA/Met Office {level_desc} {type_desc} Alert — "
            f"South East England (current as of {now_utc.strftime('%Y-%m-%d %H:%M')} UTC). "
            f"Elevated medication risk signals: {sigs_str}. {med_note}"
        )

    defaults["med_risk_signals"] = med_risk
    defaults["fep_uplift"]       = uplift
    defaults["layer4_context"]   = ctx

    return defaults


print("Fetching UKHSA Weather-Health Alert...")
weather_alert = fetch_weather_alert()

print(f"  Status:        {weather_alert['api_status']}")
print(f"  Alert level:   {weather_alert['alert_level'].upper()}")
print(f"  Alert type:    {weather_alert['alert_type']}")
print(f"  FEP uplift:    +{weather_alert['fep_uplift']} points (applied to high/critical districts)")
print(f"  Med risk:      {weather_alert['med_risk_signals'] or 'none'}")
if weather_alert['layer4_context']:
    print(f"\n  Layer 4 context string:")
    print(f"  {weather_alert['layer4_context']}")
else:
    print(f"  Layer 4 context: none (green alert)")

## Cell 3 — Fetch NHS Fingertips Indicators
10 real outcomes indicators for Kent & Medway ICB via `fingertips_py`.
Unchanged from v3.

In [ ]:
FINGERTIPS_INDICATORS = {
    'falls_65':           (22401, 'Falls admissions 65+',        KENT_ICB_ONS),
    'falls_65_79':        (22402, 'Falls admissions 65-79',      KENT_ICB_ONS),
    'falls_80':           (22403, 'Falls admissions 80+',        KENT_ICB_ONS),
    'winter_mortality':   (90360, 'Winter mortality index',      KENT_COUNTY),
    'loneliness':         (94175, 'Loneliness often/always',     KENT_ICB_ONS),
    'social_isolation':   (90280, 'Social isolation - SC users', KENT_COUNTY),
    'dementia_diagnosis': (92949, 'Dementia diagnosis rate 65+', KENT_ICB_ONS),
    'hip_fractures_65':   (41401, 'Hip fractures 65+',           KENT_ICB_ONS),
    'hip_fractures_80':   (41403, 'Hip fractures 80+',           KENT_ICB_ONS),
    'fuel_poverty':       (93759, 'Fuel poverty',                KENT_ICB_ONS),
}

fingertips_results = {}
print("Fetching NHS Fingertips indicators...")
print(f"{'Indicator':<35} {'Kent':>8}  {'England':>8}  Period")
print("-" * 70)

for key, (ind_id, label, area_code) in FINGERTIPS_INDICATORS.items():
    try:
        data = ftp.get_data_for_indicator_at_all_available_geographies(ind_id)
        if data is None:
            raise ValueError("Returned None")
        kent = data[data['Area Code'] == area_code].sort_values('Time period')
        eng  = data[data['Area Code'] == ENGLAND].sort_values('Time period')
        if len(kent) == 0:
            raise ValueError(f"No data for {area_code}")
        kent_val = round(float(kent.tail(1)['Value'].values[0]), 2)
        eng_val  = round(float(eng.tail(1)['Value'].values[0]), 2) if len(eng) else None
        period   = str(kent.tail(1)['Time period'].values[0])
        fingertips_results[key] = {
            'value': kent_val, 'england': eng_val,
            'period': period, 'source': f'NHS Fingertips indicator {ind_id}', 'label': label,
        }
        direction = "▲" if eng_val and kent_val > eng_val else "▼"
        print(f"  {label:<33} {kent_val:>8}  {str(eng_val):>8}  {period}  {direction}")
    except Exception as e:
        print(f"  FAILED {label:<29} -- {e}")
        fingertips_results[key] = {
            'value': None, 'england': None, 'period': None,
            'source': f'NHS Fingertips indicator {ind_id}', 'label': label,
        }

real_ft = sum(1 for v in fingertips_results.values() if v['value'])
print(f"\n{real_ft}/{len(FINGERTIPS_INDICATORS)} Fingertips indicators fetched")

## Cell 4 — Stream NHSBSA EPD at Practice Level
Streams the EPD CSV line by line (~18M rows).
Captures 7 BNF signal groups at district level via practice postcode.
Unchanged from v3. Takes 3–5 minutes.

In [ ]:
TARGET_BNF = (
    '0401010',   # Hypnotics
    '0401020',   # Anxiolytics
    '0403',      # Antidepressants (incl anticholinergic tricyclics)
    '060602',    # Bisphosphonates
    '0201',      # Diuretics
    '0205',      # ACE inhibitors / ARBs
    '0704010',   # Bladder antimuscarinics
)

def bnf_to_signal(bnf):
    if   bnf.startswith('0401010'): return 'hypnotics'
    elif bnf.startswith('0401020'): return 'anxiolytics'
    elif bnf.startswith('0403'):    return 'antidepressants'
    elif bnf.startswith('060602'):  return 'bisphosphonates'
    elif bnf.startswith('0201'):    return 'diuretics'
    elif bnf.startswith('0205'):    return 'ace_arb'
    elif bnf.startswith('0704010'): return 'bladder_antimusc'
    return None

district_signal_items = defaultdict(lambda: defaultdict(int))
district_signal_qty   = defaultdict(lambda: defaultdict(float))
signal_items    = defaultdict(int)
signal_quantity = defaultdict(float)
signal_drugs    = defaultdict(set)
practice_district_cache = {}
unmapped_practices = set()
unmapped_items = 0
rows_read = kent_rows = mapped_rows = 0
buffer = ""; header_done = False

print(f"Streaming NHSBSA EPD — {EPD_PERIOD} (v4.0 practice-level mode)")
print(f"Targeting {len(TARGET_BNF)} BNF groups. Takes 3–5 minutes.\n")

try:
    with requests.get(EPD_URL, stream=True, timeout=300,
                      headers={"User-Agent": "Mozilla/5.0 AssistivSystems/4.0"}) as resp:
        print(f"HTTP {resp.status_code}")
        if resp.status_code != 200:
            raise ValueError(f"HTTP {resp.status_code} — check EPD_URL in Cell 1")
        for raw_chunk in resp.iter_content(chunk_size=512*1024):
            buffer += raw_chunk.decode('utf-8', errors='replace')
            lines = buffer.split('\n'); buffer = lines[-1]
            for line in lines[:-1]:
                line = line.strip()
                if not line: continue
                if not header_done:
                    header_done = True; continue
                rows_read += 1
                if rows_read % 2_000_000 == 0:
                    print(f"  {rows_read:>12,} rows | {kent_rows:>6,} Kent | "
                          f"{mapped_rows:>6,} mapped | {len(practice_district_cache)} practices seen")
                if KENT_FILTER not in line: continue
                try:
                    row = next(csv.reader([line]))
                except Exception:
                    continue
                if len(row) <= max(ICB_I, BNF_I, ITEMS_I, QTY_I, PRACTICE_I, POSTCODE_I): continue
                if KENT_FILTER not in row[ICB_I]: continue
                bnf = row[BNF_I].strip('"')
                if not bnf.startswith(TARGET_BNF): continue
                signal = bnf_to_signal(bnf)
                if not signal: continue
                try:
                    items         = int(row[ITEMS_I])
                    qty           = float(row[QTY_I])
                    chem          = row[CHEM_I]
                    practice_code = row[PRACTICE_I].strip()
                    postcode      = row[POSTCODE_I].strip().upper()
                except (ValueError, IndexError):
                    continue
                kent_rows += 1
                signal_items[signal]    += items
                signal_quantity[signal] += qty
                signal_drugs[signal].add(chem)
                if practice_code not in practice_district_cache:
                    outward  = postcode.split()[0] if postcode else ''
                    district = POSTCODE_TO_DISTRICT.get(outward)
                    practice_district_cache[practice_code] = district
                    if district is None:
                        unmapped_practices.add(f"{practice_code}({outward})")
                district = practice_district_cache[practice_code]
                if district:
                    district_signal_items[district][signal] += items
                    district_signal_qty[district][signal]   += qty
                    mapped_rows += 1
                else:
                    unmapped_items += items
except Exception as e:
    import traceback; print(f"\nError at row {rows_read:,}: {e}"); traceback.print_exc()

print(f"\nCOMPLETE — {rows_read:,} rows scanned, {kent_rows:,} Kent matched")
print(f"  District-mapped: {mapped_rows:,} ({100*mapped_rows/max(kent_rows,1):.1f}%)")
print(f"  Unmapped items:  {unmapped_items:,}")
print(f"  Practices:       {len(practice_district_cache)} seen | {len(unmapped_practices)} unmapped")
if unmapped_practices:
    print(f"  Unmapped list: {sorted(unmapped_practices)[:20]}")

## Cell 5 — Calculate District-Level Prescribing Rates
Converts raw item counts to rates per 1,000 patients using district-specific list sizes.
Unchanged from v3. QA flags implausible district ratios.

In [ ]:
EPD_SIGNAL_KEYS = [
    'hypnotics', 'anxiolytics', 'antidepressants',
    'bisphosphonates', 'diuretics', 'ace_arb', 'bladder_antimusc'
]

epd_results = {}
print("ICB-level rates (for comparison):")
print(f"  {'Signal':<22} {'Kent/1k':>9}  {'Eng/1k':>9}  {'Ratio':>7}")
print(f"  {'-'*55}")
for key in EPD_SIGNAL_KEYS:
    items  = signal_items.get(key, 0)
    qty    = signal_quantity.get(key, 0.0)
    drugs  = len(signal_drugs.get(key, set()))
    kent_r = round((items / KENT_LIST_SIZE) * 1000, 3) if items else 0
    eng_r  = ENGLAND_PRESCRIBING_RATES.get(key, 0)
    ratio  = round(kent_r / eng_r, 3) if eng_r and kent_r else 0
    estimate_flag = " [est]" if key in ('anxiolytics', 'bladder_antimusc') else ""
    epd_results[key] = {
        'rate_per_1000': kent_r, 'england': eng_r, 'items': items,
        'qty': round(qty), 'substances': drugs, 'ratio': ratio,
        'period': EPD_PERIOD,
        'source': f'NHSBSA EPD EPD_SNOMED_{EPD_PERIOD.replace(" ","").upper()}',
        'england_note': 'literature estimate' if estimate_flag else 'OpenPrescribing 2024/25',
    }
    print(f"  {key:<22} {kent_r:>9.3f}  {eng_r:>9.1f}{estimate_flag}  {ratio:>7.3f}")

print(f"\nDistrict prescribing rates per 1,000 patients:")
header = f"  {'District':<25} {'List':>7}"
for s in EPD_SIGNAL_KEYS:
    header += f"  {s[:7]:>8}"
print(header); print(f"  {'-'*110}")

epd_district_results = {}
for district in ALL_DISTRICTS:
    list_size = DISTRICT_LIST_SIZES[district]
    epd_district_results[district] = {}
    row_str = f"  {district:<25} {list_size:>7,}"
    for key in EPD_SIGNAL_KEYS:
        items  = district_signal_items[district].get(key, 0)
        rate   = round((items / list_size) * 1000, 3) if items else 0
        eng_r  = ENGLAND_PRESCRIBING_RATES.get(key, 0)
        ratio  = round(rate / eng_r, 3) if eng_r and rate else 0
        epd_district_results[district][key] = {
            'rate_per_1000': rate, 'england': eng_r,
            'items': items, 'ratio': ratio,
            'period': EPD_PERIOD, 'source': f'NHSBSA EPD practice-level — {EPD_PERIOD}',
        }
        flag = "▲" if ratio > 1.05 else "▼" if ratio < 0.95 else "="
        row_str += f"  {rate:>7.2f}{flag}"
    print(row_str)

print("\nQA — district signals with ratio > 2.0 or < 0.3 vs England:")
issues = []
for district in ALL_DISTRICTS:
    for key in EPD_SIGNAL_KEYS:
        r = epd_district_results[district][key].get('ratio', 0)
        if r > 2.0 or (0 < r < 0.3):
            issues.append(f"  *** {district} / {key}: ratio {r:.3f} — review list size or BNF filter")
if issues:
    for i in issues: print(i)
else:
    print("  All district rates within plausible range ✓")

real_epd = sum(1 for v in epd_results.values() if v['rate_per_1000'] > 0)
print(f"\n{real_epd}/{len(EPD_SIGNAL_KEYS)} EPD signals computed")

## Cell 6 — Build District FEP Scores & Commit to GitHub
Normalises signals to 0–100, computes weighted FEP scores, commits `kent-fep-data.json`.

**v4.0 changes:**
- Weather alert status and context injected into each district record
- `fep_weather` field added alongside base `fep` — weather-uplifted score for high/critical districts
- `weather_alert` block added to JSON root for Layer 4 consumption
- `data_quality` metadata corrected: 14 real signals, 3 modelled (synthetic)
- Scorecard compares v4.0 base vs v4.0 weather-uplifted scores

In [ ]:
# ── SIGNAL DEFINITIONS ────────────────────────────────────────────────────────
SIGNAL_NAMES = [
    "Over-75s Living Alone",       # 0   synthetic (modelled from ONS Census)
    "Falls Admissions 65+",        # 1   Fingertips real
    "Hip Fracture Rate 65+",       # 2   Fingertips real
    "Deprivation (IMD)",           # 3   synthetic (modelled from IMD 2019)
    "Winter Mortality Index",      # 4   Fingertips real
    "Care Home Gap",               # 5   synthetic (modelled from CQC register)
    "Loneliness Rate",             # 6   Fingertips real
    "Dementia Diagnosis Rate",     # 7   Fingertips real (inverted — lower = worse)
    "Hip Fractures 80+",           # 8   Fingertips real
    "Social Isolation Rate",       # 9   Fingertips real
    "Hypnotics Prescribing",       # 10  EPD district-level
    "Antidepressant Rate",         # 11  EPD district-level
    "Bisphosphonate Rate",         # 12  EPD district-level
    "Diuretics Rate",              # 13  EPD district-level
    "ACE/ARB Prescribing",         # 14  EPD district-level
    "Anxiolytics Prescribing",     # 15  EPD district-level
    "Bladder Antimuscarinic Rate", # 16  EPD district-level
]

EPD_SIGNAL_INDICES      = list(range(10, 17))
EPD_SIGNAL_KEYS_ORDERED = [
    'hypnotics', 'antidepressants', 'bisphosphonates',
    'diuretics', 'ace_arb', 'anxiolytics', 'bladder_antimusc'
]

WEIGHTS = [
    0.13,  # alone_75
    0.12,  # falls_65
    0.09,  # hip_65
    0.08,  # deprivation
    0.08,  # winter_mortality
    0.07,  # care_home_gap
    0.06,  # loneliness
    0.06,  # dementia
    0.05,  # hip_80
    0.05,  # social_isolation
    0.05,  # hypnotics
    0.05,  # antidepressants
    0.04,  # bisphosphonates
    0.01,  # diuretics
    0.01,  # ace_arb
    0.03,  # anxiolytics
    0.02,  # bladder_antimusc
]
assert abs(sum(WEIGHTS) - 1.0) < 0.001, f"Weights sum to {sum(WEIGHTS):.4f} — must be 1.0"
print(f"Weight check: {sum(WEIGHTS):.4f} ✓")

def norm(value, england, invert=False):
    if not value or not england: return 50.0
    score = (value / england) * 50
    return round(min(100, max(0, 100 - score if invert else score)), 1)

def ft(key, invert=False):
    v = fingertips_results.get(key, {})
    return norm(v.get('value'), v.get('england'), invert)

def epd_for_district(district, signal_key):
    v = epd_district_results.get(district, {}).get(signal_key, {})
    return norm(v.get('rate_per_1000'), ENGLAND_PRESCRIBING_RATES.get(signal_key))

ICB_BASE_NONEEPD = [
    50.0,
    ft('falls_65'),
    ft('hip_fractures_65'),
    50.0,
    ft('winter_mortality'),
    50.0,
    ft('loneliness'),
    ft('dementia_diagnosis', invert=True),
    ft('hip_fractures_80'),
    ft('social_isolation'),
    0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
]

print("\nICB normalised scores for non-EPD signals (England avg = 50):")
for i, (name, score) in enumerate(zip(SIGNAL_NAMES[:10], ICB_BASE_NONEEPD[:10])):
    is_real = score != 50.0
    tag = "REAL  " if is_real else "MODELLED"
    bar = ">" if score > 50 else "<" if score < 50 else "="
    print(f"  {bar} {score:>5.1f}  [{tag}]  {name}")

PROFILES = {
    "Thanet":              [1.30,1.25,1.20,1.35,1.28,1.30,1.25,1.18,1.20,1.22, 1,1,1,1,1,1,1],
    "Folkestone & Hythe":  [1.22,1.18,1.15,1.22,1.20,1.20,1.18,1.12,1.15,1.15, 1,1,1,1,1,1,1],
    "Dover":               [1.18,1.15,1.12,1.18,1.15,1.10,1.14,1.08,1.10,1.10, 1,1,1,1,1,1,1],
    "Swale":               [1.12,1.10,1.08,1.12,1.10,1.12,1.08,1.05,1.08,1.05, 1,1,1,1,1,1,1],
    "Medway":              [1.06,1.05,1.04,1.08,1.05,1.08,1.02,1.02,1.05,1.02, 1,1,1,1,1,1,1],
    "Gravesham":           [1.00,0.98,1.02,1.02,1.00,1.05,0.98,1.00,1.02,1.00, 1,1,1,1,1,1,1],
    "Ashford":             [0.96,0.95,0.98,0.98,0.96,1.00,0.94,0.96,0.98,0.95, 1,1,1,1,1,1,1],
    "Canterbury":          [0.90,0.90,0.92,0.85,0.92,0.92,0.92,0.90,0.90,0.90, 1,1,1,1,1,1,1],
    "Dartford":            [0.88,0.88,0.90,0.95,0.88,0.90,0.88,0.88,0.88,0.88, 1,1,1,1,1,1,1],
    "Maidstone":           [0.85,0.85,0.88,0.95,0.85,0.92,0.85,0.85,0.85,0.85, 1,1,1,1,1,1,1],
    "Tonbridge & Malling": [0.78,0.78,0.82,0.78,0.80,0.85,0.80,0.80,0.80,0.80, 1,1,1,1,1,1,1],
    "Sevenoaks":           [0.65,0.65,0.68,0.52,0.65,0.75,0.65,0.65,0.65,0.65, 1,1,1,1,1,1,1],
    "Tunbridge Wells":     [0.58,0.58,0.62,0.58,0.60,0.65,0.60,0.60,0.60,0.60, 1,1,1,1,1,1,1],
}

LAD_CODES = {
    "Thanet":"E07000114", "Folkestone & Hythe":"E07000112", "Dover":"E07000108",
    "Swale":"E07000113",  "Medway":"E06000035",  "Gravesham":"E07000109",
    "Ashford":"E07000105","Canterbury":"E07000106","Dartford":"E07000107",
    "Maidstone":"E07000110","Tonbridge & Malling":"E07000115",
    "Sevenoaks":"E07000111","Tunbridge Wells":"E07000116",
}

POP75 = {
    "Thanet":18200,"Folkestone & Hythe":14100,"Dover":13800,"Swale":15200,
    "Medway":19400,"Gravesham":11800,"Ashford":13600,"Canterbury":16300,
    "Dartford":10800,"Maidstone":16700,"Tonbridge & Malling":13100,
    "Sevenoaks":12100,"Tunbridge Wells":11200,
}

# ── v4.0: Weather uplift config ────────────────────────────────────────────────
# Applied to districts scoring 'high' or 'critical' during amber/red alerts.
# fep_weather is stored alongside fep (base) — the map can display either.
# The uplift is not baked into weights; it is additive and labelled separately.
w_level  = weather_alert["alert_level"]
w_type   = weather_alert["alert_type"]
w_uplift = weather_alert["fep_uplift"]
w_active = w_level in ("yellow", "amber", "red")

if w_active:
    print(f"\n⚠ Weather alert active: {w_level.upper()} {w_type} — "
          f"+{w_uplift} FEP uplift on high/critical districts")
    print(f"  Elevated medication risk: {weather_alert['med_risk_signals']}")
else:
    print(f"\nWeather: {w_level.upper()} — no uplift applied")

# ── BUILD DISTRICT SCORES ──────────────────────────────────────────────────────
districts = []
print(f"\nDistrict FEP scores (v4.0):")
print(f"  {'District':<25} {'FEP':>5}  {'W-FEP':>6}  {'Risk':<10}  antidep  hypnot  anxiol")
print("  " + "-" * 80)

for name, profile in PROFILES.items():
    signals = [
        round(min(100, max(0, ICB_BASE_NONEEPD[i] * profile[i])), 1)
        for i in range(17)
    ]
    for idx, epd_key in zip(EPD_SIGNAL_INDICES, EPD_SIGNAL_KEYS_ORDERED):
        signals[idx] = epd_for_district(name, epd_key)

    fep_base = round(min(100, max(0, sum(s * w for s, w in zip(signals, WEIGHTS)))))
    risk     = "critical" if fep_base >= 70 else "high" if fep_base >= 55 else "moderate" if fep_base >= 40 else "low"

    # v4.0: weather uplift applied only to high/critical districts during active alerts
    fep_weather = fep_base
    if w_active and risk in ("high", "critical"):
        fep_weather = min(100, fep_base + w_uplift)

    # Recalculate displayed risk using weather-uplifted score for map display
    risk_weather = "critical" if fep_weather >= 70 else "high" if fep_weather >= 55 else "moderate" if fep_weather >= 40 else "low"

    ad = epd_district_results[name]['antidepressants']['rate_per_1000']
    hy = epd_district_results[name]['hypnotics']['rate_per_1000']
    ax = epd_district_results[name]['anxiolytics']['rate_per_1000']
    uplift_marker = f" +{w_uplift}⚠" if fep_weather > fep_base else ""
    print(f"  {name:<25} {fep_base:>5}  {fep_weather:>6}{uplift_marker:<4}  {risk:<10}  {ad:>7.2f}  {hy:>6.2f}  {ax:>6.2f}")

    districts.append({
        "name":          name,
        "lad_code":      LAD_CODES[name],
        "fep":           fep_base,         # base score — stable, weather-independent
        "fep_weather":   fep_weather,      # weather-uplifted score — use for live map display
        "risk":          risk,             # based on fep_base
        "risk_weather":  risk_weather,     # based on fep_weather
        "signals":       signals,
        "signal_names":  SIGNAL_NAMES,
        "pop75":         POP75[name],
        "list_size":     DISTRICT_LIST_SIZES[name],
        "epd_district":  {k: epd_district_results[name][k] for k in EPD_SIGNAL_KEYS},
    })

districts.sort(key=lambda x: x["fep_weather"], reverse=True)

# ── Scorecard vs previous commit ───────────────────────────────────────────────
print("\n── v4.0 vs previous committed scores ──")
try:
    prev = requests.get(
        f"https://raw.githubusercontent.com/{GITHUB_REPO}/main/{GITHUB_FILE}",
        timeout=10
    ).json()
    prev_scores = {d['name']: d['fep'] for d in prev.get('districts', [])}
    print(f"  {'District':<25} {'prev':>5}  {'v4.0':>5}  {'Δ':>5}  {'W-FEP':>6}")
    print(f"  {'-'*55}")
    for d in districts:
        n  = d['name']
        v2 = prev_scores.get(n, '?')
        v4 = d['fep']
        vw = d['fep_weather']
        delta = (v4 - v2) if isinstance(v2, int) else '?'
        print(f"  {n:<25} {str(v2):>5}  {v4:>5}  {str(delta):>5}  {vw:>6}")
except Exception as e:
    print(f"  (Could not fetch previous scores: {e})")

# ── ASSEMBLE OUTPUT ────────────────────────────────────────────────────────────
real_signals = (
    [k for k, v in fingertips_results.items() if v.get('value')] +
    [k for k, v in epd_results.items() if v.get('rate_per_1000', 0) > 0]
)

output = {
    "meta": {
        "generated":          datetime.now(timezone.utc).isoformat(),
        "description":        "Kent & Medway FEP scores — Assistiv Systems Layer 2",
        "version":            "4.0 — weather-health alert integration",
        "icb":                "NHS Kent and Medway ICB (QKS)",
        "icb_ons_code":       KENT_ICB_ONS,
        # CORRECTED: 14 real signals, 3 modelled — not '17 real'
        "signals_real_count": len(real_signals),
        "signals_modelled_count": 3,
        "data_quality":       f"{len(real_signals)} real signals · 3 modelled (synthetic) of 17 total",
        "signals_real":       real_signals,
        "signals_modelled":   ["alone_75", "deprivation_imd", "care_home_gap"],
        "signal_names":       SIGNAL_NAMES,
        "weights":            WEIGHTS,
        "sources": {
            "fingertips":     "NHS Fingertips/OHID PHOF via fingertips_py",
            "epd":            f"NHSBSA EPD practice-level — {EPD_PERIOD}",
            "epd_method":     "Practice POSTCODE (col 13) → district via POSTCODE_TO_DISTRICT",
            "demographic":    "ONS Census 2021 · IMD 2019 · CQC register",
            "weather":        "UKHSA Weather-Health Alerting / Met Office",
        },
        "new_in_v4": [
            "UKHSA/Met Office Weather-Health Alert fetched at run time",
            "fep_weather field: weather-uplifted FEP for live map display",
            "weather_alert block added to JSON root for Layer 4 context injection",
            "data_quality metadata corrected: 14 real, 3 modelled (was incorrectly '17 real')",
            "Scratch cells (ambulance/emergency search) removed from production notebook",
        ],
        "note": (
            "fep = base score (weather-independent, use for longitudinal comparison). "
            "fep_weather = live score including weather uplift (use for operational map display). "
            "EPD: district-level actual. Fingertips + modelled: ICB × demographic profile. "
            "Phase 2: MSOA Fingertips linkage."
        ),
    },
    # v4.0: weather_alert block at JSON root — consumed by Layer 4 RESILIENCE
    "weather_alert": weather_alert,
    "icb_baseline": {
        "fingertips":  fingertips_results,
        "prescribing": epd_results,
    },
    "districts": districts,
}

# ── COMMIT TO GITHUB ───────────────────────────────────────────────────────────
def commit_to_github(content, repo, filepath, token):
    api_url = f"https://api.github.com/repos/{repo}/contents/{filepath}"
    headers = {"Authorization": "token " + token,
               "Accept": "application/vnd.github.v3+json"}
    b64 = base64.b64encode(json.dumps(content, indent=2).encode()).decode()
    r = requests.get(api_url, headers=headers)
    sha = r.json().get("sha") if r.status_code == 200 else None
    w_tag = f" [{w_level.upper()} {w_type}]" if w_active else ""
    payload = {
        "message": (
            f"FEP v4.0 — {datetime.now(timezone.utc).strftime('%Y-%m-%d')} — "
            f"practice-level EPD ({len(practice_district_cache)} practices){w_tag}"
        ),
        "content": b64,
        "branch":  "main",
    }
    if sha: payload["sha"] = sha
    r = requests.put(api_url, headers=headers, json=payload)
    if r.status_code in (200, 201):
        print(f"\nCommitted to GitHub ✓")
        print(f"  File:   https://raw.githubusercontent.com/{GITHUB_REPO}/main/{GITHUB_FILE}")
        print(f"  Commit: {r.json().get('commit',{}).get('html_url','')}")
        return True
    print(f"\nCommit failed: {r.status_code} — {r.json().get('message','')}")
    return False

print("\nCommitting to GitHub...")
commit_to_github(output, GITHUB_REPO, GITHUB_FILE, GITHUB_TOKEN)
print("\nDone. Run quarterly for EPD refresh; re-run Cell 2 alone for weather update.")
print(f"Practices mapped: {len(practice_district_cache)} | Unmapped: {len(unmapped_practices)}")
print(f"Weather: {w_level.upper()} {w_type} | FEP uplift applied: {'+'+str(w_uplift) if w_active else 'none'}")